# pandas·Excel: 필터와 결측값 정책을 명시하기

공학프로그래밍입문 · 2026학년도 1학기 · 핵심 활동 약 40~50분

**목표는 문법 암기보다 원리를 이해하고, 코딩 AI에 정확한 질문·지시를 하며, 결과를 검토하는 능력입니다.**
코드는 예측한 뒤 실행하고, 값·조건을 직접 바꾸며 확인합니다. AI 도구를 사용할 수 없으면 제시된 가상 답변을 동료와 검토합니다.

준비: 리스트·딕셔너리·함수, pandas 기초. 패키지: pandas, openpyxl

## 오늘 설명할 수 있어야 하는 것

- 행·열 선택과 결측 처리 정책을 설명한다.
- 표의 통계와 Excel 왕복 저장 결과를 독립적으로 검증한다.


## 1. 먼저 원리를 읽기

DataFrame은 이름 붙은 열과 행 인덱스를 갖는 표입니다. `.loc`는 레이블·조건, `.iloc`는 위치를 사용합니다.
평균을 구할 때 분모는 전체 행 수인지 유효 관측 수인지 정해야 합니다. 여기서는 결측을 제외하며 무강수 0은 포함합니다. 아래 표는 수업용 합성 데이터입니다.


## 2. 실행 전 예측하기

A 지점 `[0, 6, 결측]`의 유효 관측 수와 평균은 얼마일까요? 결측을 0으로 채우면 평균은 어떻게 달라질까요?


아래 기록을 먼저 채운 뒤 다음 코드 셀을 실행하세요. 정답과 다르면 예측에 사용한 가정을 고칩니다.

- 내 예상 결과:
- 그렇게 생각한 이유:
- 실행 뒤 발견한 차이:


In [ ]:
import pandas as pd
import math

observations = pd.DataFrame({
    "station": ["A", "A", "A", "B"],
    "rain_mm": [0.0, 6.0, None, 2.0],
})
selected = observations.loc[observations["station"] == "A", "rain_mm"]
print(observations)
print("전체 행:", len(selected), "유효 관측:", selected.count(), "평균:", selected.mean())
assert selected.count() == 2 and selected.mean() == 3
assert selected.fillna(0).mean() == 2


### 관찰을 원리와 연결하기

결측값을 0으로 바꾸는 것은 데이터 정리 문법만의 문제가 아니라 분석 가정의 변경입니다. `.count()`와 `len()`을 비교하고, 포함된 원자료 몇 행을 직접 확인하세요.


다음 셀의 입력이나 조건을 하나 바꾸세요. 바꾼 값, 예상한 변화, 실제 결과를 적습니다. 단순 재실행만으로 이 활동을 마치지 않습니다.


In [ ]:
from pathlib import Path
import tempfile

excel_dir = Path(tempfile.mkdtemp(prefix="ce100-excel-"))
excel_path = excel_dir / "rainfall-practice.xlsx"
# Colab에는 일반적으로 설치되어 있습니다. 없다면 안내에 따라 openpyxl을 설치하세요.
observations.to_excel(excel_path, index=False, engine="openpyxl")
restored = pd.read_excel(excel_path, engine="openpyxl")
pd.testing.assert_frame_equal(observations, restored, check_dtype=False)
print("저장한 Excel:", excel_path)
print(restored)
# dtype까지 같다는 계약은 아닙니다. 열·값·결측 위치를 우선 확인합니다.


## 3. 작은 기능을 직접 완성하기

`station_mean(frame, station)`은 `station`, `rain_mm` 열이 있는 표와 지점 이름을 받아 해당 지점의 유효 관측 평균을 `float`로 반환합니다.
해당 지점이 없거나 전부 결측이면 `None`을 반환합니다. 관측값은 비음수 유한 숫자 또는 결측으로 한정하며 원본 표를 바꾸지 않습니다.


먼저 입력·출력 예를 하나 더 직접 만드세요. 모르는 문법은 도움말이나 AI에 질문해도 됩니다. AI 코드 전체를 받기 전에 필요한 처리 단계를 자신의 말로 2~3줄 적으세요.

- 내가 정한 추가 입력과 예상 출력:
- 처리 단계:


In [ ]:
def station_mean(frame, station):
    # TODO: 지점 선택, 결측 제외, 유효 관측이 없는 경우를 나누세요.
    raise NotImplementedError("평균의 분모와 결측 정책을 구현하세요")


### 공개 점검

아래 점검 셀을 실행합니다. **“미완료”는 통과가 아닙니다.** 위 함수를 구현한 뒤 다시 실행하세요.
실패하면 기대값을 바꾸기보다 입력·계약·구현 중 무엇이 다른지 조사합니다. 공개 시험만 통과했다고 모든 입력에 옳다는 뜻은 아닙니다.


In [ ]:
def check_answer(candidate):
    frame = pd.DataFrame({"station": ["A", "A", "A", "B", "C"],
                          "rain_mm": [0., 6., None, 2., None]})
    before = frame.copy(deep=True)
    assert math.isclose(candidate(frame, "A"), 3.)
    assert math.isclose(candidate(frame, "B"), 2.)
    assert candidate(frame, "C") is None and candidate(frame, "X") is None
    pd.testing.assert_frame_equal(frame, before)
    print("공개 점검 통과: 지점 선택·0·결측·빈 선택·원본 보존")

try:
    check_answer(station_mean)
except NotImplementedError:
    print("미완료: 위 TODO를 구현한 뒤 이 셀을 다시 실행하세요. 아직 점검을 통과하지 않았습니다.")


## 4. AI 제안을 검토하고 수정 지시하기

다음은 오류를 검토하기 위해 만든 **가상 AI 답변**입니다. 실제 AI 응답이나 정답이 아닙니다.
실행 전에 어떤 요구사항을 놓쳤는지 찾고, 반례가 그 문제를 어떻게 드러내는지 설명하세요.


In [ ]:
def ai_suggestion(frame, station):
    selected = frame.loc[frame["station"] == station, "rain_mm"]
    return selected.fillna(0).mean()

print("가상 제안 평균:", ai_suggestion(observations, "A"))
assert ai_suggestion(observations, "A") != 3


### 검토 질문

이 제안은 결측을 관측된 0으로 취급합니다. 숫자가 그럴듯해도 분모가 다릅니다. 표의 선택 조건, 단위, 결측 정책을 포함해 수정 요청을 쓰세요.


- 위 제안이 어긴 요구사항:
- 오류를 드러내는 입력과 기대 결과:
- 한 문장 수정 지시:


### 코딩 AI에게 보낼 요청 작성

입력 열 이름과 예시 5행, rain_mm 단위, 지점 필터, 결측 제외·0 포함, 유효 관측이 없으면 None, 원본 표 보존을 명시한다. 손계산과 Excel 저장 후 재읽기 검증도 요청한다.


위 문장을 그대로 복사하는 대신 오늘의 실제 입력·기대값·실패 조건을 넣어 요청문을 완성하세요.
아래 틀로 **구현 전 계획**을 먼저 요청하고, 받은 코드에는 새 반례를 하나 추가합니다.

```text
목적과 환경: Python / Google Colab에서 ... 하려고 한다.
입력: 자료형·구조·단위는 ... 이며, 예시는 ... 이다.
출력: ... 형식으로 ... 를 반환해야 한다.
조건: ... 는 포함하고 ... 는 제외한다. 잘못된 입력은 ... 로 처리한다.
검증: 정상 예 ... / 경계 예 ... / 반례 ... 의 기대 결과는 ... 이다.
먼저 모호한 조건을 짚고 처리 계획을 설명해 줘. 그 뒤 짧은 코드와 검증을 제시해 줘.
```

- 내가 보낸 최초 요청:
- 검토 후 보낸 수정 요청:
- AI 제안에서 채택·수정·거절한 부분과 근거:
- 내가 추가한 시험의 입력·기대값·실제값:


## 5. 새 상황에 옮겨 보기

일별 관측 수가 다른 두 지점의 평균을 다시 평균하면 전체 관측 평균과 같을까요? 작은 표로 반례를 만드세요.


이 활동은 한 가지 구현을 암기했는지가 아니라 요구사항이 달라졌을 때 무엇을 다시 질문하는지 확인합니다.


## 마무리 기록

- 오늘 사용한 핵심 원리를 자신의 말로 한 문장:
- 결과를 신뢰할 근거(시험 입력·기대값·실제값):
- 아직 확인하지 않은 조건 또는 적용 한계:
- AI 없이 설명할 수 있는 코드 한 줄과 그 역할:

**완료 기준:** 예측을 남겼고, 입력·조건을 바꿨고, TODO를 구현해 공개 점검을 통과했고, 새 반례와 수정 지시를 작성했으며, 결과의 한계를 설명할 수 있습니다.

[과목의 실습 방식과 점검 기준](../LEARNING_GUIDE.md)을 참고하세요.


## 선택 확장

필요하면 Colab의 별도 셀에서 `%pip install openpyxl`을 실행합니다. CSV·JSON·Excel 중 형식을 고를 때 열 자료형, 결측 표현, 날짜 보존을 비교하세요.


## 출처와 개편 범위

    기존 강의 노트북의 주제를 바탕으로 핵심 예제, 합성 연습 데이터, AI 검토 활동을 새로 구성했습니다.
    원본 파일명은 유지했고, 원본 Drive 링크와 개편 전·후 해시는 강의 폴더의 `materials-manifest.json`에 기록합니다.
    노트북 실행에 외부 AI API나 원본 로컬 ZIP은 필요하지 않습니다. 학습용 오류 제안은 반례를 확인하는 코드이며, 미완성 TODO와 구분합니다.
